# Training
Hyperparameter and augmentation live in the script `pipeline/train.py`, so a run started here and a run started from the CLI are the same run.

Run `make download` first if `datasets/` is empty.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import torch

from pipeline.train import AUGMENTATION, train

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device      ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("augmentation", AUGMENTATION)

## Smoke Test

One epoch on 5% of the data. Proves the dataset, the model download, and the output paths all work before committing to a long run.

`workers=0` is deliberate. A Jupyter kernel on Windows can hang with dataloader workers above zero.

In [ ]:
results = train(
    epochs=1,
    fraction=0.05,
    batch=2,
    device=DEVICE,
    workers=0,
    name="smoke_test",
    exist_ok=True,
)

print("\nsaved to", results.save_dir)

## Real run

Tune `batch` to the GPU. On 6 GB start at 8 and drop to 4 if it runs out of memory, or pass `batch=-1` to let Ultralytics fit it.

This is long. A kernel disconnect kills it, so for a full run prefer the terminal:

```bash
make train NAME="<your-run-name> ARGS="<additional-args>
```

In [ ]:
results = train(
    epochs=100,
    batch=-1,
    device=DEVICE,
    workers=0,
    name="ppe_v1",
)

## Results

Per-class numbers matter more than the headline mAP here. The dataset carries 25 classes and only a handful are PPE, so a good average can hide a weak `Hardhat`.

In [ ]:
import pandas as pd

run_dir = Path(results.save_dir)
history = pd.read_csv(run_dir / "results.csv")
history.columns = history.columns.str.strip()
history.tail()

In [ ]:
from IPython.display import Image, display

for plot in (
    "results.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "val_batch0_pred.jpg",
):
    path = run_dir / plot
    if path.exists():
        print(plot)
        display(Image(filename=str(path), width=900))
    else:
        print(f"{plot}: not generated for this run")